In [34]:
# some imports 
import numpy as np 
import networkx as nx
import copy 
from itertools import product
from collections import Counter
import matplotlib.pyplot as plt
from numpy import random
#D-Wave Ocean SDK related
import dimod 
from dwave.system import DWaveSampler
from dwave.samplers import SimulatedAnnealingSampler
# from dwave.cloud import Client
# client = Client.from_config()
from dwave.embedding import embed_ising, unembed_sampleset
from minorminer import find_embedding
import pickle
from pathlib import Path

seed = 1224
random.seed(seed)

In [35]:
embeddings = pickle.loads(Path("../pkls/embeddings.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("../pkls/SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("../pkls/QA_sample_sets.pkl").read_bytes())
sa_results = pickle.loads(Path("../pkls/sa_results.pkl").read_bytes())
qa_results = pickle.loads(Path("../pkls/qa_results.pkl").read_bytes())
graphs = pickle.loads(Path("../pkls/graphs.pkl").read_bytes())
pt4_results = pickle.loads(Path("../pkls/pt4_results.pkl").read_bytes())

In [36]:
labels = ["Uniform Regular","Normal Regular","Binary Regular",
          "Uniform Erdos-Renyi","Normal Erdos-Renyi","Binary Erdos-Renyi",
          "Uniform Full","Normal Full","Binary Full",
          "Uniform Square","Normal Square","Binary Square",
          "Uniform Barabasi-Albert (1)","Normal Barabasi-Albert (1)","Binary Barabasi-Albert (1)",
          "Uniform Barabasi-Albert (2)","Normal Barabasi-Albert (2)","Binary Barabasi-Albert (2)"]

In [37]:
print(len(sa_results))
print(qa_results)

18
[(np.float64(-744.0140747708512), np.float64(-723.8470972702349)), (np.float64(-1200.3909071644514), np.float64(-1137.6404698568474)), (np.float64(-1320.0), np.float64(-1294.324)), (np.float64(-585.8286390950403), np.float64(-468.43625987958063)), (np.float64(-1072.183595816818), np.float64(-849.3015996396745)), (np.float64(-1071.0), np.float64(-853.173)), (np.float64(-831.9442122219914), np.float64(-674.8230328185494)), (np.float64(-1485.0814771630307), np.float64(-1170.1579559451081)), (np.float64(-1502.0), np.float64(-1171.66)), (np.float64(-2035.9919468103803), np.float64(-2013.514406513445)), (np.float64(-3285.126382629163), np.float64(-3188.720914222981)), (np.float64(-3695.0), np.float64(-3665.19)), (np.float64(-2402.1387108978165), np.float64(-2365.5360718797906)), (np.float64(-3787.8717745430877), np.float64(-3666.062332698338)), (np.float64(-4087.0), np.float64(-4015.358)), (np.float64(-858.4827008393061), np.float64(-810.3673379351094)), (np.float64(-1338.874828751965), n

In [38]:
import numpy as np

def print_latex_table(sa, qa, labels, caption="Simulated vs. quantum annealing results.",
                      label="tab:sa-vs-qa", precision=2, row_header="Instance"):
    """
    sa, qa : lists of (best, average) pairs, e.g. [(np.float64(-774.2), np.float64(-768.9)), ...]
    labels : list of row labels, one per pair, in the same order as sa and qa
    Prints a LaTeX table with one row per label.
    """
    if not (len(sa) == len(qa) == len(labels)):
        raise ValueError(
            f"Length mismatch: sa={len(sa)}, qa={len(qa)}, labels={len(labels)}"
        )

    fmt = f"{{:.{precision}f}}"

    def escape(s):
        # Escape LaTeX special characters in labels
        for ch, rep in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"),
                        ("$", r"\$"), ("#", r"\#"), ("_", r"\_"),
                        ("{", r"\{"), ("}", r"\}")]:
            s = s.replace(ch, rep)
        return s

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \begin{tabular}{l rr rr}")
    lines.append(r"        \toprule")
    lines.append(r"        & \multicolumn{2}{c}{\textbf{Simulated Annealing}} "
                 r"& \multicolumn{2}{c}{\textbf{Quantum Annealing}} \\")
    lines.append(r"        \cmidrule(lr){2-3} \cmidrule(lr){4-5}")
    lines.append(rf"        \textbf{{{row_header}}} & \textbf{{Lowest}} & \textbf{{Average}} "
                 r"& \textbf{Lowest} & \textbf{Average} \\")
    lines.append(r"        \midrule")

    for lab, (sa_best, sa_avg), (qa_best, qa_avg) in zip(labels, sa, qa):
        lines.append(
            f"        {escape(str(lab))} & "
            f"{fmt.format(float(sa_best))} & {fmt.format(float(sa_avg))} & "
            f"{fmt.format(float(qa_best))} & {fmt.format(float(qa_avg))} \\\\"
        )

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")

    print("\n".join(lines))

print_latex_table(sa_results, qa_results, labels)

\begin{table}[H]
    \centering
    \caption{Simulated vs. quantum annealing results.}
    \label{tab:sa-vs-qa}
    \begin{tabular}{l rr rr}
        \toprule
        & \multicolumn{2}{c}{\textbf{Simulated Annealing}} & \multicolumn{2}{c}{\textbf{Quantum Annealing}} \\
        \cmidrule(lr){2-3} \cmidrule(lr){4-5}
        \textbf{Instance} & \textbf{Lowest} & \textbf{Average} & \textbf{Lowest} & \textbf{Average} \\
        \midrule
        Uniform Regular & -774.21 & -768.87 & -744.01 & -723.85 \\
        Normal Regular & -1311.06 & -1297.66 & -1200.39 & -1137.64 \\
        Binary Regular & -1344.00 & -1343.58 & -1320.00 & -1294.32 \\
        Uniform Erdos-Renyi & -688.35 & -681.63 & -585.83 & -468.44 \\
        Normal Erdos-Renyi & -1205.84 & -1183.17 & -1072.18 & -849.30 \\
        Binary Erdos-Renyi & -1235.00 & -1219.22 & -1071.00 & -853.17 \\
        Uniform Full & -984.02 & -971.09 & -831.94 & -674.82 \\
        Normal Full & -1684.68 & -1662.71 & -1485.08 & -1170.16 \\
        Bi

In [39]:
import numpy as np

def _escape(s):
    for ch, rep in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"),
                    ("$", r"\$"), ("#", r"\#"), ("_", r"\_"),
                    ("{", r"\{"), ("}", r"\}")]:
        s = s.replace(ch, rep)
    return s


def print_latex_diff_table(sa, qa, labels,
                           caption="Differences between simulated and quantum annealing, sorted by lowest-energy difference.",
                           label="tab:sa-qa-diff", precision=2, row_header="Instance",
                           sort_by="d_best"):
    """
    sa, qa : lists of (best, average) pairs
    labels : list of row labels, same order as sa and qa
    sort_by: which column orders the rows from high to low:
             "d_best"  (SA - QA, Lowest)
             "d_avg"   (SA - QA, Average)
             "gap_sa"  (SA Average - Lowest)
             "gap_qa"  (QA Average - Lowest)

    Columns:
      SA - QA (Lowest)       = sa_best - qa_best
      SA - QA (Average)      = sa_avg  - qa_avg
      SA gap (Avg - Lowest)  = sa_avg  - sa_best
      QA gap (Avg - Lowest)  = qa_avg  - qa_best
    """
    if not (len(sa) == len(qa) == len(labels)):
        raise ValueError(
            f"Length mismatch: sa={len(sa)}, qa={len(qa)}, labels={len(labels)}"
        )
    if sort_by not in ("d_best", "d_avg", "gap_sa", "gap_qa"):
        raise ValueError(f"Unknown sort_by: {sort_by!r}")

    def f(x):
        # Math mode so negative values get a proper minus sign
        return f"${float(x):.{precision}f}$"

    rows = []
    for lab, (sa_best, sa_avg), (qa_best, qa_avg) in zip(labels, sa, qa):
        rows.append({
            "label": str(lab),
            "d_best": float(sa_best) - float(qa_best),
            "d_avg":  float(sa_avg) - float(qa_avg),
            "gap_sa": float(sa_avg) - float(sa_best),
            "gap_qa": float(qa_avg) - float(qa_best),
        })

    # High to low; sorted() is stable, so ties keep the original label order
    rows = sorted(rows, key=lambda r: r[sort_by], reverse=True)

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \begin{tabular}{l rr rr}")
    lines.append(r"        \toprule")
    lines.append(r"        & \multicolumn{2}{c}{\textbf{SA $-$ QA}} "
                 r"& \multicolumn{2}{c}{\textbf{Average $-$ Lowest}} \\")
    lines.append(r"        \cmidrule(lr){2-3} \cmidrule(lr){4-5}")
    lines.append(rf"        \textbf{{{row_header}}} & \textbf{{Lowest}} & \textbf{{Average}} "
                 r"& \textbf{SA} & \textbf{QA} \\")
    lines.append(r"        \midrule")

    for r in rows:
        lines.append(
            f"        {_escape(r['label'])} & {f(r['d_best'])} & {f(r['d_avg'])} & "
            f"{f(r['gap_sa'])} & {f(r['gap_qa'])} \\\\"
        )

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")

    print("\n".join(lines))


print_latex_diff_table(sa_results, qa_results, labels)                    # sorted by SA - QA (Lowest)
# print_latex_diff_table(sa_results, qa_results, labels, sort_by="d_avg")  # or by another column

\begin{table}[H]
    \centering
    \caption{Differences between simulated and quantum annealing, sorted by lowest-energy difference.}
    \label{tab:sa-qa-diff}
    \begin{tabular}{l rr rr}
        \toprule
        & \multicolumn{2}{c}{\textbf{SA $-$ QA}} & \multicolumn{2}{c}{\textbf{Average $-$ Lowest}} \\
        \cmidrule(lr){2-3} \cmidrule(lr){4-5}
        \textbf{Instance} & \textbf{Lowest} & \textbf{Average} & \textbf{SA} & \textbf{QA} \\
        \midrule
        Binary Square & $-18.00$ & $-30.68$ & $17.13$ & $29.81$ \\
        Binary Regular & $-24.00$ & $-49.25$ & $0.42$ & $25.68$ \\
        Uniform Square & $-24.34$ & $-33.42$ & $13.40$ & $22.48$ \\
        Uniform Regular & $-30.19$ & $-45.02$ & $5.34$ & $20.17$ \\
        Uniform Barabasi-Albert (2) & $-43.71$ & $-78.34$ & $13.49$ & $48.12$ \\
        Binary Barabasi-Albert (2) & $-58.00$ & $-117.83$ & $17.99$ & $77.82$ \\
        Uniform Barabasi-Albert (1) & $-63.36$ & $-84.77$ & $15.19$ & $36.60$ \\
        Binary Barab

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # write files without needing a display
import matplotlib.pyplot as plt


def plot_sa_vs_qa(sa_results, qa_results, labels, outdir=".", basename="sa_vs_qa",
                  baseline=0.0, formats=("png", "pdf"), group_size=3):
    """
    sa_results, qa_results : lists of (best, expected) pairs, same order as labels
    baseline   : value where the bars start. 0 by default; set it closer to the data
                 (e.g. -700) to magnify small differences.
    group_size : a faint vertical separator is drawn after every `group_size` labels
    Writes <basename>_best, <basename>_expected, and <basename>_combined files.
    The y axis is inverted, so more negative (better) results give taller bars.
    """
    if not (len(sa_results) == len(qa_results) == len(labels)):
        raise ValueError(f"Length mismatch: sa={len(sa_results)}, qa={len(qa_results)}, labels={len(labels)}")

    sa = np.array([[float(b), float(e)] for b, e in sa_results])  # columns: best, expected
    qa = np.array([[float(b), float(e)] for b, e in qa_results])

    n = len(labels)
    x = np.arange(n)
    width = 0.38
    colors = {"SA": "tab:blue", "QA": "tab:orange"}

    def draw(ax, col, title):
        # bottom=baseline with height=value-baseline makes each bar span baseline -> value
        ax.bar(x - width / 2, sa[:, col] - baseline, width, bottom=baseline,
               label="Simulated annealing", color=colors["SA"], edgecolor="black", linewidth=0.5)
        ax.bar(x + width / 2, qa[:, col] - baseline, width, bottom=baseline,
               label="Quantum annealing", color=colors["QA"], edgecolor="black", linewidth=0.5)

        # Faint separators between groups (e.g. Uniform / Normal / Binary of one graph type)
        for b in np.arange(group_size, n, group_size) - 0.5:
            ax.axvline(b, color="gray", alpha=0.35, linewidth=1.0, zorder=0)

        lo = min(sa[:, col].min(), qa[:, col].min())
        pad = 0.03 * abs(lo - baseline)
        ax.set_ylim(baseline, lo - pad)          # baseline at the bottom, more negative at the top
        ax.set_xlim(-0.5, n - 0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Energy (higher bar = lower energy)")
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.legend()

    def save(fig, name):
        for fmt in formats:
            fig.savefig(f"{outdir}/{name}.{fmt}", dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Separate figures: best and expected
    for col, key, title in ((0, "best", "Best (lowest) energy"),
                            (1, "expected", "Expected (average) energy")):
        fig, ax = plt.subplots(figsize=(12, 5.5))
        draw(ax, col, f"Simulated vs. quantum annealing: {title}")
        fig.tight_layout()
        save(fig, f"{basename}_{key}")

    # Combined figure: both panels stacked
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    draw(axes[0], 0, "Best (lowest) energy")
    draw(axes[1], 1, "Expected (average) energy")
    fig.suptitle("Simulated vs. quantum annealing")
    fig.tight_layout()
    save(fig, f"{basename}_combined")

plot_sa_vs_qa(sa_results, qa_results, labels)